In [ ]:
import numpy as np
import pandas as pd


In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv('../datasets/final_dataset_feature_selection_v6.csv')


In [4]:
df.head()


,property_type,sector,bedRoom,bathroom,balcony,agePossession,property_id,builtup_area,pooja room,store room,servant room,furnishing_type,facility_category,floor_category,price
0,1.0,32.0,5,4,3.0,2.0,2447.0,1350.0,0,0,1,0,1.0,2.0,4.35
1,0.0,59.0,3,4,4.0,0.0,1006.0,2450.0,0,0,1,2,0.0,2.0,3.10
2,0.0,74.0,2,2,2.0,4.0,853.0,978.0,0,0,0,0,1.0,1.0,1.65
3,0.0,109.0,3,4,4.0,1.0,2866.0,1964.0,0,0,1,0,1.0,2.0,2.00
4,0.0,17.0,4,5,4.0,3.0,2548.0,2996.0,0,0,1,1,0.0,2.0,3.90


In [5]:
X = df.drop(columns=['price'])
y = df['price']


In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [ ]:
# cols to perform one-hot encoding on
columns_to_encode = ['sector', 'balcony', 'agePossession', 'furnishing_type', 'facility_category', 'floor_category']


In [9]:
# since right skewed --> log transformation
y_transformed = np.log1p(y)


In [15]:
# column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['property_type', 'bedRoom', 'bathroom', 'builtup_area', 'servant room', 'store room', 'pooja room']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), columns_to_encode)
    ],
    remainder='passthrough'
)


In [16]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


In [17]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')


d:\Projects\capstone-project\myenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
d:\Projects\capstone-project\myenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [18]:
scores.mean()


np.float64(0.8613781189520004)

In [20]:
scores.std()


np.float64(0.019246255879519072)

In [21]:
X_train,  X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)


In [22]:
pipeline.fit(X_train, y_train)


d:\Projects\capstone-project\myenv\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['property_type', 'bedRoom',
                                                   'bathroom', 'builtup_area',
                                                   'servant room', 'store room',
                                                   'pooja room']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector', 'balcony',
                                                   'agePossession',
                                                   'furnishing_type',
                                                   'facility_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [23]:
y_pred = pipeline.predict(X_test)


In [24]:
y_pred = np.expm1(y_pred)  # inverse log transformation to get back to original scale


In [25]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test), y_pred)


0.6647464804194968

making mistakes, need to fine tune.

In [27]:
from sklearn.svm import SVR


In [28]:
# let's try another one
preprocessor2 = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['property_type', 'bedRoom', 'bathroom', 'builtup_area', 'servant room', 'store room', 'pooja room']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), columns_to_encode)
    ],
    remainder='passthrough'
)


In [31]:
pipeline_svr = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
])


In [32]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline_svr, X, y_transformed, cv=kfold, scoring='r2')


d:\Projects\capstone-project\myenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
d:\Projects\capstone-project\myenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [33]:
scores.mean()


np.float64(-0.036801842911096026)